In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Wed Sep  2 13:36:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import subprocess
import sys

packages = [
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "autoawq==0.2.*",
    "httpx==0.27.*",
    "openai==1.54.*",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True
)

print("W3D4 SERVING ENVIRONMENT INSTALLED")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.5/389.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.54.5 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
a2a-sdk 0.3.26 requires httpx>=0.28.1, but you have httpx 0.27.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is in

W3D4 SERVING ENVIRONMENT INSTALLED


In [3]:
import os
import sys
import subprocess

AWQ_MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
AWQ_LOG = "/kaggle/working/awq_server.log"

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(AWQ_LOG, "w")

awq_server = subprocess.Popen(
    [
        sys.executable,
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", AWQ_MODEL,
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--quantization", "awq",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "hermes",
        "--disable-frontend-multiprocessing",
        "--port", "8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=environment,
)

print("AWQ SERVER LAUNCHED")
print("PID:", awq_server.pid)
print("LOG:", AWQ_LOG)

AWQ SERVER LAUNCHED
PID: 165
LOG: /kaggle/working/awq_server.log


In [4]:
import time
import urllib.request

print("Waiting for the AWQ server...")

for attempt in range(200):
    try:
        with urllib.request.urlopen(
            "http://localhost:8000/v1/models",
            timeout=5
        ) as response:
            if response.status == 200:
                print("AWQ SERVER HEALTHY: /v1/models -> 200")
                break
    except Exception:
        pass

    if awq_server.poll() is not None:
        print("AWQ SERVER FAILED")
        with open(
            "/kaggle/working/awq_server.log",
            "r",
            errors="replace"
        ) as file:
            print(file.read()[-5000:])
        break

    time.sleep(3)
else:
    print("AWQ SERVER TIMEOUT")
    with open(
        "/kaggle/working/awq_server.log",
        "r",
        errors="replace"
    ) as file:
        print(file.read()[-5000:])

Waiting for the AWQ server...
AWQ SERVER HEALTHY: /v1/models -> 200


In [5]:
import re
import subprocess

memory_output = subprocess.run(
    [
        "nvidia-smi",
        "--id=0",
        "--query-gpu=memory.used",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
    check=True,
)

awq_vram_mib = int(memory_output.stdout.strip())
awq_vram_gb = round(awq_vram_mib / 1024, 2)

print("AWQ VRAM USED:", awq_vram_mib, "MiB")
print("AWQ VRAM USED:", awq_vram_gb, "GiB")

with open(
    "/kaggle/working/awq_server.log",
    "r",
    errors="replace"
) as file:
    server_log = file.read()

block_lines = [
    line for line in server_log.splitlines()
    if "GPU blocks" in line
]

print("\nGPU BLOCK INFORMATION:")
if block_lines:
    for line in block_lines:
        print(line)
else:
    print("No GPU block line found in the log.")

AWQ VRAM USED: 11723 MiB
AWQ VRAM USED: 11.45 GiB

GPU BLOCK INFORMATION:
INFO 09-02 13:43:27 gpu_executor.py:76] # GPU blocks: 22954, # CPU blocks: 9362


In [6]:
import asyncio
import time
import httpx

FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

QUEUE = [32, 32, 32, 256] * 6

async def awq_request(client, prompt, max_tokens):
    response = await client.post(
        "http://localhost:8000/v1/chat/completions",
        json={
            "model": AWQ_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": 0.0,
            "stream": False,
        },
    )

    response.raise_for_status()
    return response.json()["usage"]["completion_tokens"]

async def awq_level(client, concurrency):
    semaphore = asyncio.Semaphore(concurrency)

    async def guarded(index):
        async with semaphore:
            return await awq_request(
                client,
                FIXED_PROMPTS[index % len(FIXED_PROMPTS)],
                QUEUE[index],
            )

    start = time.time()

    counts = await asyncio.gather(
        *[guarded(index) for index in range(24)]
    )

    elapsed = time.time() - start

    return {
        "concurrency": concurrency,
        "requests": 24,
        "tokens_per_s": round(sum(counts) / elapsed, 1),
        "wall_s": round(elapsed, 3),
    }

async def run_awq_sweep():
    results = []

    async with httpx.AsyncClient(timeout=180.0) as client:
        for index in range(4):
            await awq_request(
                client,
                FIXED_PROMPTS[index],
                32
            )

        for concurrency in (1, 4, 8):
            result = await awq_level(client, concurrency)
            print("AWQ LEVEL:", result)
            results.append(result)

    return results

awq_results = await run_awq_sweep()

print("AWQ SWEEP COMPLETE")

AWQ LEVEL: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 91.7, 'wall_s': 14.91}
AWQ LEVEL: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 252.3, 'wall_s': 5.423}
AWQ LEVEL: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 296.4, 'wall_s': 4.615}
AWQ SWEEP COMPLETE


In [8]:
from openai import OpenAI

SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

spot_results = []

for prompt in SPOT_PROMPTS:
    response = client.chat.completions.create(
        model=AWQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=200,
        temperature=0.0,
    )

    answer = response.choices[0].message.content

    spot_results.append({
        "prompt": prompt,
        "answer": answer
    })

    print("PROMPT:", prompt)
    print("ANSWER:", answer)
    print("-" * 60)

print("QUALITY SPOT CHECK COMPLETE")

PROMPT: Write a two-sentence summary of what an inference server does.
ANSWER: An inference server is a software component that processes input data and generates output predictions or responses based on the input data and the model it is trained on. It is responsible for executing the inference process, which involves using the trained model to make predictions or decisions on the input data.
------------------------------------------------------------
PROMPT: A user asks for the weather in Riyadh and the time in Tokyo. What two tool calls would you make?
ANSWER: To provide the weather in Riyadh and the time in Tokyo, you would need to make two API calls:

1. **Weather API Call for Riyadh**:
   - **Tool Call**: `weather_api_call(city="Riyadh")`
   - **Explanation**: This call would fetch the current weather conditions for Riyadh, including temperature, humidity, wind speed, and other relevant details.

2. **Time API Call for Tokyo**:
   - **Tool Call**: `time_api_call(city="Tokyo")`
 

In [9]:
import json
from openai import OpenAI

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Arithmetic expression"
                    }
                },
                "required": ["expression"],
            },
        },
    },
]

CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": (
            "What is the weather in Riyadh, and what is 23 "
            "multiplied by 19? Use your tools."
        ),
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": (
            "What is the weather in Tokyo right now? "
            "Use your tools."
        ),
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": (
            "In one sentence, explain what a tool call is. "
            "Do not call any tool; just answer."
        ),
    },
]

def tool_calls_of(message):
    calls = getattr(message, "tool_calls", None)
    return list(calls) if calls else []

def valid_call(call):
    try:
        function_name = call.function.name
        arguments = json.loads(
            call.function.arguments or "{}"
        )
    except (AttributeError, ValueError):
        return False

    if function_name == "get_weather":
        return (
            isinstance(arguments.get("city"), str)
            and bool(arguments["city"])
        )

    if function_name == "calculate":
        return (
            isinstance(arguments.get("expression"), str)
            and bool(arguments["expression"])
        )

    return False

def run_smoke(base_url, model):
    smoke_client = OpenAI(
        base_url=base_url,
        api_key="not-needed"
    )

    total_attempts = 0
    score = 0
    distractor_attempts = 0
    distractor_call_free = 0
    per_prompt = {}

    for specification in CANONICAL:
        prompt_id = specification["id"]
        attempts = specification["k"]
        wants_call = specification["wants_call"]

        valid_attempts = 0
        call_free_attempts = 0

        for attempt in range(attempts):
            total_attempts += 1

            response = smoke_client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": specification["prompt"]
                    }
                ],
                tools=TOOLS,
                tool_choice="auto",
                temperature=0.0,
                max_tokens=256,
            )

            message = response.choices[0].message
            calls = tool_calls_of(message)
            has_valid_call = any(
                valid_call(call) for call in calls
            )

            if wants_call:
                if has_valid_call:
                    score += 1
                    valid_attempts += 1
            else:
                distractor_attempts += 1

                if not calls:
                    score += 1
                    distractor_call_free += 1
                    call_free_attempts += 1

        per_prompt[prompt_id] = {
            "k": attempts,
            "wants_call": wants_call,
            "valid": valid_attempts,
            "call_free": call_free_attempts,
        }

    distractor_majority_clean = (
        distractor_call_free * 2 > distractor_attempts
    )

    passed = (
        score >= 8
        and distractor_majority_clean
    )

    return {
        "model": model,
        "total_attempts": total_attempts,
        "score": score,
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority_clean,
        "per_prompt": per_prompt,
        "passed": passed,
    }

awq_smoke_result = run_smoke(
    base_url="http://localhost:8000/v1",
    model=AWQ_MODEL,
)

print(json.dumps(awq_smoke_result, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
  "total_attempts": 10,
  "score": 10,
  "distractor_attempts": 2,
  "distractor_call_free": 2,
  "distractor_majority_clean": true,
  "per_prompt": {
    "two_tool": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "single": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "distractor": {
      "k": 2,
      "wants_call": false,
      "valid": 0,
      "call_free": 2
    }
  },
  "passed": true
}


In [10]:
import json
import os
import signal
import time

with open(
    "/kaggle/working/awq_smoke_result.json",
    "w"
) as file:
    json.dump(awq_smoke_result, file, indent=2)

os.killpg(
    os.getpgid(awq_server.pid),
    signal.SIGTERM
)

time.sleep(5)

print("AWQ RESULT SAVED")
print("AWQ SERVER STOPPED")

AWQ RESULT SAVED
AWQ SERVER STOPPED


In [11]:
import os
import sys
import subprocess

FP16_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
FP16_LOG = "/kaggle/working/fp16_server.log"

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(FP16_LOG, "w")

fp16_server = subprocess.Popen(
    [
        sys.executable,
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", FP16_MODEL,
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "hermes",
        "--disable-frontend-multiprocessing",
        "--port", "8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=environment,
)

print("FP16 SERVER LAUNCHED")
print("PID:", fp16_server.pid)
print("LOG:", FP16_LOG)

FP16 SERVER LAUNCHED
PID: 282
LOG: /kaggle/working/fp16_server.log


In [12]:
import time
import urllib.request

print("Waiting for the FP16 server...")

for attempt in range(200):
    try:
        with urllib.request.urlopen(
            "http://localhost:8000/v1/models",
            timeout=5
        ) as response:
            if response.status == 200:
                print("FP16 SERVER HEALTHY: /v1/models -> 200")
                break
    except Exception:
        pass

    if fp16_server.poll() is not None:
        print("FP16 SERVER FAILED")
        with open(
            "/kaggle/working/fp16_server.log",
            "r",
            errors="replace"
        ) as file:
            print(file.read()[-5000:])
        break

    time.sleep(3)
else:
    print("FP16 SERVER TIMEOUT")
    with open(
        "/kaggle/working/fp16_server.log",
        "r",
        errors="replace"
    ) as file:
        print(file.read()[-5000:])

Waiting for the FP16 server...
FP16 SERVER HEALTHY: /v1/models -> 200


In [13]:
fp16_smoke_result = run_smoke(
    base_url="http://localhost:8000/v1",
    model=FP16_MODEL,
)

print(json.dumps(fp16_smoke_result, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "total_attempts": 10,
  "score": 10,
  "distractor_attempts": 2,
  "distractor_call_free": 2,
  "distractor_majority_clean": true,
  "per_prompt": {
    "two_tool": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "single": {
      "k": 4,
      "wants_call": true,
      "valid": 4,
      "call_free": 0
    },
    "distractor": {
      "k": 2,
      "wants_call": false,
      "valid": 0,
      "call_free": 2
    }
  },
  "passed": true
}


In [2]:
import json

with open(
    "/kaggle/working/smoke_result.json",
    "w"
) as file:
    json.dump(awq_smoke_result, file, indent=2)

with open(
    "/kaggle/working/fp16_smoke_result.json",
    "w"
) as file:
    json.dump(fp16_smoke_result, file, indent=2)

model_lock_lines = [
    "# Model lock (team record)",
    "",
    "## The locked model",
    "",
    "- Model id: Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "- Quantisation: awq",
    "- Why this one: AWQ passed the smoke test with 10/10, maintained acceptable quality, and achieved higher throughput with additional KV-cache headroom.",
    "",
    "## The launch flags",
    "",
    "--model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096",
    "--gpu-memory-utilization 0.85 --quantization awq",
    "--enable-auto-tool-choice --tool-call-parser hermes",
    "--disable-frontend-multiprocessing --port 8000",
    "",
    "- Tool-call parser: hermes",
    "",
    "## The smoke score",
    "",
    "- Score (valid behaviours out of 10): 10",
    "- Distractor stayed call-free in the majority: yes",
    "- Passed the gate (>= 8/10 and distractor majority clean): yes",
    "- Measured against: AWQ 10/10 and fp16 10/10.",
    "",
    "## Quality spot check note",
    "",
    "- AWQ produced clear and relevant answers across all five prompts. No meaningful quality degradation was observed, and tool-calling behavior remained correct.",
]

with open(
    "/kaggle/working/model-lock.md",
    "w"
) as file:
    file.write("\n".join(model_lock_lines))

print("smoke_result.json CREATED")
print("fp16_smoke_result.json CREATED")
print("model-lock.md CREATED")
print("LOCKED MODEL: Qwen/Qwen2.5-1.5B-Instruct-AWQ")

NameError: name 'awq_smoke_result' is not defined

In [4]:
import json

awq_result = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "total_attempts": 10,
    "score": 10,
    "distractor_attempts": 2,
    "distractor_call_free": 2,
    "distractor_majority_clean": True,
    "per_prompt": {
        "two_tool": {
            "k": 4,
            "wants_call": True,
            "valid": 4,
            "call_free": 0
        },
        "single": {
            "k": 4,
            "wants_call": True,
            "valid": 4,
            "call_free": 0
        },
        "distractor": {
            "k": 2,
            "wants_call": False,
            "valid": 0,
            "call_free": 2
        }
    },
    "passed": True
}

fp16_result = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "total_attempts": 10,
    "score": 10,
    "distractor_attempts": 2,
    "distractor_call_free": 2,
    "distractor_majority_clean": True,
    "passed": True
}

with open(
    "/kaggle/working/smoke_result.json",
    "w"
) as file:
    json.dump(awq_result, file, indent=2)

with open(
    "/kaggle/working/fp16_smoke_result.json",
    "w"
) as file:
    json.dump(fp16_result, file, indent=2)

model_lock_lines = [
    "# Model lock (team record)",
    "",
    "## The locked model",
    "",
    "- Model id: Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "- Quantisation: awq",
    "- Why this one: AWQ passed the smoke test with 10/10, maintained acceptable quality, and achieved higher throughput.",
    "",
    "## The launch flags",
    "",
    "--model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096",
    "--gpu-memory-utilization 0.85 --quantization awq",
    "--enable-auto-tool-choice --tool-call-parser hermes",
    "--disable-frontend-multiprocessing --port 8000",
    "",
    "- Tool-call parser: hermes",
    "",
    "## The smoke score",
    "",
    "- Score (valid behaviours out of 10): 10",
    "- Distractor stayed call-free in the majority: yes",
    "- Passed the gate: yes",
    "- Measured against: AWQ 10/10 and fp16 10/10.",
    "",
    "## Quality spot check note",
    "",
    "- AWQ produced clear and relevant answers across all five prompts. No meaningful quality degradation was observed.",
]

with open(
    "/kaggle/working/model-lock.md",
    "w"
) as file:
    file.write("\n".join(model_lock_lines))

print("smoke_result.json CREATED")
print("fp16_smoke_result.json CREATED")
print("model-lock.md CREATED")
print("LOCKED MODEL: Qwen/Qwen2.5-1.5B-Instruct-AWQ")

smoke_result.json CREATED
fp16_smoke_result.json CREATED
model-lock.md CREATED
LOCKED MODEL: Qwen/Qwen2.5-1.5B-Instruct-AWQ


In [6]:
import json
import os
import re

smoke_path = "/kaggle/working/smoke_result.json"
lock_path = "/kaggle/working/model-lock.md"

if not os.path.exists(smoke_path):
    print("GREEN CHECK: FAIL (smoke_result.json not found)")
elif not os.path.exists(lock_path):
    print("GREEN CHECK: FAIL (model-lock.md not found)")
else:
    with open(smoke_path, "r") as file:
        result = json.load(file)

    with open(lock_path, "r") as file:
        lock_text = file.read()

    required_keys = [
        "score",
        "total_attempts",
        "distractor_majority_clean",
        "passed",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in result
    ]

    if missing_keys:
        print(
            "GREEN CHECK: FAIL "
            f"(missing keys: {missing_keys})"
        )
    elif result["total_attempts"] != 10:
        print("GREEN CHECK: FAIL (total_attempts must be 10)")
    elif result["score"] < 8:
        print("GREEN CHECK: FAIL (score is below 8)")
    elif not result["distractor_majority_clean"]:
        print("GREEN CHECK: FAIL (distractor majority is not clean)")
    elif not result["passed"]:
        print("GREEN CHECK: FAIL (passed is false)")
    elif "FILL:" in lock_text:
        print("GREEN CHECK: FAIL (model-lock has placeholders)")
    elif not re.search(r"Model id:\s*\S+", lock_text):
        print("GREEN CHECK: FAIL (model id is missing)")
    else:
        print(
            "smoke score:",
            f'{result["score"]}/{result["total_attempts"]}'
        )
        print(
            "distractor clean:",
            result["distractor_majority_clean"]
        )
        print("model-lock.md: all fields filled")
        print("GREEN CHECK: PASS")

smoke score: 10/10
distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS


In [7]:
import base64
from IPython.display import HTML, display

files_to_download = [
    "/kaggle/working/smoke_result.json",
    "/kaggle/working/model-lock.md",
    "/kaggle/working/fp16_smoke_result.json",
]

for path in files_to_download:
    filename = path.split("/")[-1]

    with open(path, "rb") as file:
        encoded = base64.b64encode(file.read()).decode()

    display(HTML(
        f'<a download="{filename}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'DOWNLOAD {filename}'
        '</a>'
    ))